In [1]:
import re
import numpy as np
import pandas as pd
import collections
import matplotlib.pyplot as plt
import networkx as nx
from treelib import Node, Tree
from collections import Iterable
from collections import Counter
import math
plt.rcParams['font.sans-serif']=['SimHei']
plt.rcParams['figure.figsize']=['600', '600']
plt.rcParams['axes.unicode_minus']=False

C:\Anaconda\lib\site-packages\ipykernel_launcher.py:8: DeprecationWarning: Using or importing the ABCs from 'collections' instead of from 'collections.abc' is deprecated, and in 3.8 it will stop working
  


In [2]:
class Label(object):
    def __init__(self, layer, read):
        self.layer=layer
        self.count=1
        self.read=read
        
# root here is str
def sum_up(root):
    children_list=tree.children(root)
    if len(children_list)>0:
        for i in children_list:
            tree.nodes[root].data.read+=sum_up(i.identifier)
        return tree.nodes[root].data.read
    else:
        return tree.nodes[root].data.read

In [5]:
data = pd.read_csv("detail_new.csv")
data.columns=["Auto_Index","Title","Author","Wechat_NN","Layer","Address","Gender",
                "Time","Stay","Share2Moment","Share2Friend","Read","Belong2"]
data=data.sort_values(["Author","Title"])
pair_list=[[item[0][0],item[0][1]] for item in data.groupby(["Author","Title"])]

In [8]:
f=open("CommonPath/fullpath.edgelist",'wb')
pair_id_list=[]
node_id_list=[]
ct=0 # node id counter
G = nx.Graph() # generate difussion tree for this pair
for pair in pair_list:
    author, title = pair[0], pair[1] 
    local_node_list=[]
    local_node_id_list=[]
    ## Generate tree according to the layer info. in table, and sum up the read
    # generate new node represents this pair
    node_id=ct
    node_id_list.append(node_id)
    pair_id=ct
    pair_id_list.append(pair_id)
    ct+=1
    data_1=data.loc[(data['Author'] == author) & (data['Title'] == title)]

    G.add_node(pair_id)
    layers=data_1.Layer.tolist()
    max_layer=max(layers)
    avg_layer=sum(layers) / len(layers)
    layer=1
    save_for_later=[]
    while layer<max_layer+1:
        for index, row in data_1.iterrows():
            if row.Layer==layer:
                WNN=row.Wechat_NN
                node_id=ct
                node_id_list.append(node_id)
                local_node_id_list.append(node_id)
                local_node_list.append(WNN)
                if row.Layer==1: 
                    G.add_node(node_id)
                    G.add_edge(node_id,pair_id,weight=int(row.Read))
                else: 
                    try:
                        p_id=local_node_id_list[local_node_list.index(row.Belong2)]
                        G.add_node(node_id)
                        G.add_edge(node_id,p_id,weight=int(row.Read))
                    except:
                        save_for_later.append([node_id,row.Belong2,layer,row.Read])
                ct+=1  
        layer+=1
    for nid in save_for_later:
        p_id=local_node_id_list[local_node_list.index(nid[1])]
        G.add_node(nid[0])
        G.add_edge(nid[0],p_id,weight=nid[2])

nx.write_edgelist(G, f, data=['weight'], delimiter=",")
f.close()
pair_info=pd.DataFrame(data=pair_list)
pair_id_list=pd.Series(pair_id_list,dtype='int')
pair_info["id"]=pair_id_list
pair_info.to_csv("CommonPath/fullpath.authorlist", header=None, index=False)

In [51]:
data_1

,Auto_Index,Title,Author,Wechat_NN,Layer,Address,Gender,Time,Stay,Share2Moment,Share2Friend,Read,Belong2
856,857,名匠年度感恩答谢会系列活动三《百大小区千套工地巡展》,+章珊,MJ~家装顾问-姗姗,1,湖南湘潭,女,2017-11-17 14:55:11.0,0,0,0,1,NaN
857,858,名匠年度感恩答谢会系列活动三《百大小区千套工地巡展》,+章珊,MJ~家装顾问-姗姗,1,湖南湘潭,女,2017-11-16 14:25:15.0,150,1,0,2,NaN
858,859,名匠年度感恩答谢会系列活动三《百大小区千套工地巡展》,+章珊,MJ~家装顾问-姗姗,1,湖南湘潭,女,2017-11-17 14:55:56.0,30,1,0,1,NaN
859,860,名匠年度感恩答谢会系列活动三《百大小区千套工地巡展》,+章珊,花中四君子,1,NaN,男,2017-11-22 17:14:27.0,0,0,0,1,NaN
860,861,名匠年度感恩答谢会系列活动三《百大小区千套工地巡展》,+章珊,MJ~家装顾问-姗姗,1,湖南湘潭,女,2017-11-23 10:45:06.0,0,1,0,1,NaN
927,928,名匠年度感恩答谢会系列活动三《百大小区千套工地巡展》,+章珊,a 阿虎,2,湖南湘潭,男,2017-11-16 14:36:42.0,60,0,0,1,MJ~家装顾问-姗姗
928,929,名匠年度感恩答谢会系列活动三《百大小区千套工地巡展》,+章珊,飘洋过海,2,中国 湖南 湘潭,男,2017-11-17 15:06:15.0,0,0,0,1,MJ~家装顾问-姗姗
929,930,名匠年度感恩答谢会系列活动三《百大小区千套工地巡展》,+章珊,MJ-家装顾问珊珊,2,湖南湘潭,男,2017-11-17 15:46:19.0,0,0,0,1,MJ~家装顾问-姗姗
